In [5]:
# 导入依赖
import gymnasium as gym
import collections
from tensorboardX import SummaryWriter

# 超参数
ENV_NAME = "FrozenLake-v1"      # FrozenLake 环境（v1 为 Gymnasium 版本）
GAMMA = 0.9                     # 折扣因子：未来奖励的衰减率
TEST_EPISODES = 20              # 每轮评估时跑的测试回合数

In [6]:
class Agent:
    """Agent 持有环境状态价值和动作价值表"""
    def __init__(self):
        self.env = gym.make(ENV_NAME)
        self.state, _ = self.env.reset()  # 初始状态
        self.rewards = collections.defaultdict(float)     # 记录每步奖励
        self.transits = collections.defaultdict(collections.Counter)  # 状态转移计数
        self.values = collections.defaultdict(float)      # V(s)：状态价值表

    def play_n_random_steps(self, count):
        """随机走 count 步，记录奖励和转移"""
        for _ in range(count):
            action = self.env.action_space.sample()  # 随机选动作
            new_state, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            # 记录奖励和转移次数
            self.rewards[(self.state, action, new_state)] = reward
            self.transits[(self.state, action)][new_state] += 1
            self.state = self.env.reset()[0] if done else new_state

    def calc_action_value(self, state, action):
        """计算 Q(s,a)：对每个可能的下一个状态，用 V(s') 做加权平均"""
        target_counts = self.transits[(state, action)]
        total = sum(target_counts.values())
        action_value = 0.0
        for tgt_state, count in target_counts.items():
            reward = self.rewards[(state, action, tgt_state)]
            action_value += (count / total) * (reward + GAMMA * self.values[tgt_state])
        return action_value

    def select_action(self, state):
        """贪心选择 Q 值最大的动作"""
        best_action, best_value = None, None
        for action in range(self.env.action_space.n):
            action_value = self.calc_action_value(state, action)
            if best_value is None or action_value > best_value:
                best_value = action_value
                best_action = action
        return best_action

    def play_episode(self, env):
        """用当前策略玩一回合，返回总奖励"""
        total_reward = 0.0
        state, _ = env.reset()
        while True:
            action = self.select_action(state)
            new_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            self.rewards[(state, action, new_state)] = reward
            self.transits[(state, action)][new_state] += 1
            total_reward += reward
            if done:
                break
            state = new_state
        return total_reward

    def value_iteration(self):
        """价值迭代：V(s) = max_a Q(s, a)"""
        for state in range(self.env.observation_space.n):
            state_values = [
                self.calc_action_value(state, action)
                for action in range(self.env.action_space.n)
            ]
            self.values[state] = max(state_values)

In [8]:
# 训练主循环：随机探索 → 价值迭代 → 测试 → 重复
if __name__ == "__main__":
    test_env = gym.make(ENV_NAME)                             # 测试专用环境
    agent = Agent()                                           # 创建 Agent
    writer = SummaryWriter(comment="-v-iteration")            # TensorBoard 日志

    iter_no = 0
    best_reward = 0.0
    while True:
        iter_no += 1
        # ① 随机探索：收集环境转移数据
        agent.play_n_random_steps(100)
        # ② 价值迭代：用收集的数据更新 V(s)
        agent.value_iteration()

        # ③ 测试：跑 TEST_EPISODES 回合，统计成功率
        total_reward = 0.0
        for _ in range(TEST_EPISODES):
            total_reward += agent.play_episode(test_env)
        avg_reward = total_reward / TEST_EPISODES

        # ④ 记录到 TensorBoard
        writer.add_scalar("avg_reward", avg_reward, iter_no)
        if avg_reward > best_reward:
            best_reward = avg_reward
            print(f"Iter {iter_no:3d}: Best reward updated → {best_reward:.3f}")
        else:
            print(f"Iter {iter_no:3d}: reward = {avg_reward:.3f},  best = {best_reward:.3f}")

        # ⑤ FrozenLake 认为平均奖励 ≥ 0.8 即已解决
        if avg_reward > 0.8:
            print(f"\nSolved in {iter_no} iterations!")
            break

    writer.close()
    test_env.close()

Iter   1: reward = 0.000,  best = 0.000
Iter   2: reward = 0.000,  best = 0.000
Iter   3: reward = 0.000,  best = 0.000
Iter   4: reward = 0.000,  best = 0.000
Iter   5: Best reward updated → 0.100
Iter   6: reward = 0.000,  best = 0.100
Iter   7: reward = 0.100,  best = 0.100
Iter   8: Best reward updated → 0.400
Iter   9: reward = 0.250,  best = 0.400
Iter  10: reward = 0.350,  best = 0.400
Iter  11: reward = 0.150,  best = 0.400
Iter  12: Best reward updated → 0.550
Iter  13: reward = 0.500,  best = 0.550
Iter  14: Best reward updated → 0.600
Iter  15: Best reward updated → 0.750
Iter  16: reward = 0.700,  best = 0.750
Iter  17: reward = 0.650,  best = 0.750
Iter  18: reward = 0.500,  best = 0.750
Iter  19: reward = 0.750,  best = 0.750
Iter  20: reward = 0.650,  best = 0.750
Iter  21: Best reward updated → 0.800
Iter  22: reward = 0.650,  best = 0.800
Iter  23: Best reward updated → 0.850

Solved in 23 iterations!
